In [17]:
import pandas as pd
from sklearn.linear_model import Ridge
import numpy as np
from sklearn.metrics import mean_squared_error
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")

In [13]:
from sklearn.base import BaseEstimator, TransformerMixin

class LeakSafeTargetEncoder(BaseEstimator, TransformerMixin):
    """Learns a smoothed average `price` per category from training data,
    reads `price` directly from X (not y), and maps fixed averages onto any future data."""

    def __init__(self, group_col, smoothing=10):
        self.group_col = group_col
        self.smoothing = smoothing

    def fit(self, X, y=None):
        self.global_mean_ = X["price"].mean()
        stats = X.groupby(self.group_col)["price"].agg(["mean", "count"])
        stats["smoothed"] = (stats["count"] * stats["mean"] + self.smoothing * self.global_mean_) / (stats["count"] + self.smoothing)
        self.mapping_ = stats["smoothed"]
        return self

    def transform(self, X):
        encoded = X[self.group_col].map(self.mapping_).fillna(self.global_mean_)
        return encoded.to_numpy().reshape(-1, 1)

In [14]:
test_encoder = LeakSafeTargetEncoder(group_col="brand_name", smoothing=10)
test_encoder.fit(train)

print("Global mean:", round(test_encoder.global_mean_, 2))
print("Celine smoothed avg:", round(test_encoder.mapping_.get("celine", -1), 2))

Global mean: 26.59
Celine smoothed avg: 112.08


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

NUM_COLS = ["item_condition_id", "shipping", "category_depth", "name_length",
            "desc_length", "name_word_count", "has_description", "is_branded"]
CAT_COLS = ["main_category", "sub_category", "sub_sub_category", "condition_label"]

def fillna_text(x):
    return x.fillna("")

def fillna_cat(x):
    return x.fillna("missing")

name_pipe = Pipeline([("fillna", FunctionTransformer(fillna_text)),
                       ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, max_df=0.9, stop_words="english"))])

desc_pipe = Pipeline([("fillna", FunctionTransformer(fillna_text)),
                       ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=2, max_df=0.9, stop_words="english"))])

cat_pipe = Pipeline([("fillna", FunctionTransformer(fillna_cat)),
                      ("ohe", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(transformers=[
    ("name_tfidf", name_pipe, "name"),
    ("desc_tfidf", desc_pipe, "item_description"),
    ("num_scale", StandardScaler(), NUM_COLS),
    ("cat_ohe", cat_pipe, CAT_COLS),
    ("cat_encode", LeakSafeTargetEncoder(group_col="main_category", smoothing=10), ["main_category", "price"]),
    ("brand_encode", LeakSafeTargetEncoder(group_col="brand_name", smoothing=10), ["brand_name", "price"]),
])

X_transformed = preprocessor.fit_transform(train)
print("Transformed shape:", X_transformed.shape)

Transformed shape: (39780, 25759)


In [18]:
full_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", Ridge(alpha=5.0, random_state=42))
])

y_train = train["log_price"]
y_val = val["log_price"]

full_pipeline.fit(train, y_train)
pred = full_pipeline.predict(val)

rmsle = np.sqrt(mean_squared_error(y_val, pred))
print("Pipeline val RMSLE:", round(rmsle, 4))
print("Original Day 3 val RMSLE: 0.5204")

Pipeline val RMSLE: 0.5214
Original Day 3 val RMSLE: 0.5204
